In [0]:
%run ./01_config

In [0]:
"""
03_generate_dataset.py  —  Synthetic SAP PM benchmark generator

Part of the SAP PM cost-aware maintenance optimisation pipeline.
Run order is filename order: 01 through 12.
"""

# 03 — Generate the benchmark dataset (v2.2 design)

# Writes the seven SAP PM entity extracts directly into the volume landing zone, and the
# ground-truth disclosure into a separate folder.
# What the v2.2 design adds over v2:

# Addition  -  Why
# Observable covariate effects on the hazard (manufacturer, plant, criticality)  -  Provides the learnable ranking signal RQ1 requires; unobservable frailty raises heterogeneity without raising discrimination (Section 4.2.3)
# Residual frailty at variance 0.08 (v2.1 used 0.55)  -  High frailty variance biases the marginal Weibull shape downward by ~30%, defeating parameter recovery
# Calendar PM cycles drawn independently of eta  -  Real cycles come from convention, not from the failure distribution the framework is meant to estimate; also preserves the two-directional optimization result
# Defect escalation with QMNUM_ORIG linkage  -  Ground-truth label for the cost-of-delay prioritizer (FR4 / RQ3)
# IFLOT.CRITICALITY  -  Required by the data contract (Section 2.4); drives downtime valuation
# PM03 planned-repair orders  -  Third order type the report's EN 13306 mapping already assumes
# DOWNTIME_VALUATION as its own COSS column  -  Order settlement cost excludes production loss, as in real SAP

# Runtime is a few seconds. Pure pandas on the driver — no Spark, no cluster scaling.

# Shared configuration from '01_config' is assumed to be in scope.

# Diagnostic — is there already data in the volume?

# Run this first. If extracts turn up under a different folder, move them and skip
# straight to notebook 04 instead of regenerating.

def walk(path, depth=0):
    try:
        for f in dbutils.fs.ls(path):
            print("  " * depth + f.name + (f"   {f.size:,} bytes" if not f.isDir() else ""))
            if f.isDir() and depth < 2:
                walk(f.path, depth + 1)
    except Exception as e:
        print("  " * depth + f"(cannot list {path}: {e})")

walk(VOLUME_ROOT)

# Configuration

# Every knob is here and every one is disclosed in simulation_parameters.csv.

# PM_CYCLE_CHOICES are round calendar intervals drawn independently of eta. This is
# deliberate and matters twice over. Substantively, it reflects how incumbent cycles are
# actually set — manufacturer recommendation, regulatory minimum, organisational habit —
# rather than assuming the organisation already knows each class's failure distribution,
# which is the knowledge the framework is supposed to supply. Methodologically, tying the
# cycle to eta leaves the incumbent policy accidentally near-optimal and collapses the
# two-directional result of Section 4.3.1.

# The covariate effect sizes are set so that the oracle concordance — the ceiling a
# perfectly-informed model could reach — sits above the pre-registered 0.70. Ranking signal
# must come from covariates the model can observe; unobserved frailty raises heterogeneity
# without raising discrimination, and at high variance it biases the Weibull shape downward.

import numpy as np
import pandas as pd
from datetime import datetime, timedelta

SEED = 42

OBS_START = datetime(2020, 1, 1)
OBS_END = datetime(2025, 12, 31)
COMMISSION_END = datetime(2022, 12, 31)   # all equipment commissioned inside the window: no left truncation

N_EQUI_PER_CLASS = 55                     # x 10 classes = 550 equipment (Section 4.1.1)
N_PLANTS = 6
N_LOC_PER_PLANT = 10                      # -> 60 functional locations

WEIBULL_BETA_RANGE = (1.3, 2.8)           # >1 throughout: wear-out, so a cost optimum exists
WEIBULL_ETA_RANGE = (380.0, 1100.0)       # characteristic life, days
PM_RESTORATION_RANGE = (0.30, 0.80)       # Kijima rho: 0 = bad as old, 1 = good as new
PM_CYCLE_CHOICES = [180, 270, 365, 540, 730]   # round calendar cycles, drawn INDEPENDENTLY of eta
FRAILTY_VARIANCE = 0.08                   # residual unobserved heterogeneity, gamma, mean 1

# Observable covariate effects on the log hazard. These are what makes RQ1 answerable: a model
# can only rank units by what it can see. Manufacturer and plant are recorded in EQUI/IFLOT, so
# their effects are learnable; frailty is the residual the model cannot reach.
MANUFACTURER_EFFECT_SD = 1.10             # log-hazard sd across manufacturers
PLANT_EFFECT_SD = 0.95                    # operating-context severity by plant
CRITICALITY_COEF = 0.35                   # log-hazard per criticality step above the mean
DEFECT_RATE_DAYS = 280.0                  # mean days between defect notifications per item
ESCALATION_BASE_HAZARD = 0.0035           # per-day base escalation hazard for an open defect

BREAKDOWN_COST_MULTIPLIER = 4.0
PLANNED_REPAIR_MULTIPLIER = 1.25
BASE_PREVENTIVE_COST_RANGE = (400.0, 2500.0)
DOWNTIME_PER_CRITICALITY = 0.25           # downtime valuation = repair cost x this x criticality

# Historical backlog response by priority code — the incumbent FIFO/priority policy the
# prioritizer is benchmarked against. A defect escalates when it deteriorates faster than
# the organisation gets to it.
PRIORITY_RESPONSE_DAYS = {1: 3.0, 2: 10.0, 3: 30.0, 4: 60.0}

ISO14224_CLASSES = [
    "Rotating-Pumps", "Rotating-Compressors", "Rotating-Fans",
    "Static-Vessels", "Static-HeatExchangers",
    "Electrical-Motors", "Electrical-Transformers",
    "Instrumentation-Sensors", "Mechanical-Valves", "Mechanical-Conveyors",
]

rng = np.random.default_rng(SEED)
HORIZON = (OBS_END - OBS_START).days
print(f"window {OBS_START:%Y-%m-%d} to {OBS_END:%Y-%m-%d} ({HORIZON} days), seed {SEED}")

# Class-level ground truth

CLASS_PARAMS = {}
for cls in ISO14224_CLASSES:
    beta = round(float(rng.uniform(*WEIBULL_BETA_RANGE)), 3)
    eta = round(float(rng.uniform(*WEIBULL_ETA_RANGE)), 1)
    CLASS_PARAMS[cls] = {
        "weibull_beta": beta,
        "weibull_eta_days": eta,
        "pm_restoration_rho": round(float(rng.uniform(*PM_RESTORATION_RANGE)), 3),
        "pm_cycle_days": int(rng.choice(PM_CYCLE_CHOICES)),
        "base_preventive_cost": round(float(rng.uniform(*BASE_PREVENTIVE_COST_RANGE)), 2),
        "breakdown_cost_multiplier": BREAKDOWN_COST_MULTIPLIER,
        "planned_repair_multiplier": PLANNED_REPAIR_MULTIPLIER,
        "frailty_variance": FRAILTY_VARIANCE,
        "defect_rate_days": DEFECT_RATE_DAYS,
        "escalation_base_hazard": ESCALATION_BASE_HAZARD,
    }

display(pd.DataFrame([{"EQTYP": c, **v} for c, v in CLASS_PARAMS.items()]))

# Helpers

def residual_life(beta, eta, age, rng):
    """Conditional (residual) Weibull life for an item that has survived to `age`."""
    u = rng.uniform(1e-4, 1 - 1e-4)
    age_term = (age / eta) ** beta if age > 0 else 0.0
    t_total = eta * ((age_term - np.log(1 - u)) ** (1.0 / beta))
    return max(t_total - age, 1.0)

def as_date(days):
    return OBS_START + timedelta(days=float(days))

qmel_rows, aufk_rows, afru_rows, coss_rows = [], [], [], []
ctr = {"qmnum": 900000, "aufnr": 700000, "rueck": 0}

def emit_order(auart, day, equnr, plant, qmnum, warpl, prio, cost_type,
               hours_mu, hours_sd, repair_cost, criticality):
    """Write one order + its confirmation + its settlement."""
    ctr["aufnr"] += 1
    aufnr = f"AO{ctr['aufnr']}"
    start = as_date(day)
    hours = max(1.0, float(rng.normal(hours_mu, hours_sd)))
    finish = start + timedelta(hours=hours)

    aufk_rows.append({
        "AUFNR": aufnr, "AUART": auart, "EQUNR": equnr, "QMNUM": qmnum,
        "WARPL": warpl, "WERKS": plant,
        "GSTRP": start.strftime("%Y-%m-%d"), "IDAT2": finish.strftime("%Y-%m-%d"),
        "PRIOK": prio,
        "ORDER_CLASS": "PREVENTIVE" if auart == "PM02" else "CORRECTIVE",
    })
    ctr["rueck"] += 1
    afru_rows.append({
        "RUECK": f"CNF{ctr['rueck']}", "AUFNR": aufnr,
        "BUDAT": finish.strftime("%Y-%m-%d"), "ISMNW": round(hours, 2),
        "ARBPL": f"WC-{rng.integers(1, 20)}", "PERSONNEL": f"TECH-{rng.integers(1, 40)}",
    })
    # Downtime valuation is recorded alongside, not inside, the settled order cost -
    # SAP order actuals carry labour/material/overhead, not lost production.
    downtime = round(repair_cost * DOWNTIME_PER_CRITICALITY * criticality, 2) if cost_type == "BREAKDOWN" else 0.0
    coss_rows.append({
        "OBJNR": aufnr, "AUFNR": aufnr,
        "KSTAR_LABOR": round(repair_cost * 0.35, 2),
        "KSTAR_MATERIAL": round(repair_cost * 0.55, 2),
        "KSTAR_OVERHEAD": round(repair_cost * 0.10, 2),
        "DOWNTIME_VALUATION": downtime,
        "TOTAL_ACTUAL_COST": round(repair_cost, 2),
        "COST_TYPE": cost_type,
    })
    return aufnr

# Master data: IFLOT, EQUI, MPLA

iflot_rows = []
for p in range(1, N_PLANTS + 1):
    for l in range(1, N_LOC_PER_PLANT + 1):
        iflot_rows.append({
            "TPLNR": f"P{p:02d}-LOC-{l:03d}", "PLTXT": f"Plant {p} / Area {l}",
            "FLTYP": "A", "WERKS": f"P{p:02d}", "IWERK": f"P{p:02d}",
            "CRITICALITY": int(rng.integers(1, 5)),
        })
iflot_df = pd.DataFrame(iflot_rows)
criticality_of = dict(zip(iflot_df.TPLNR, iflot_df.CRITICALITY))

# Covariate effects, centred so they do not absorb into the class baseline scale.
MANUFACTURERS = [f"Mfr-{i}" for i in range(1, 12)]
mfr_effect = {m: float(rng.normal(0, MANUFACTURER_EFFECT_SD)) for m in MANUFACTURERS}
mfr_effect = {k: v - np.mean(list(mfr_effect.values())) for k, v in mfr_effect.items()}
plant_effect = {f"P{p:02d}": float(rng.normal(0, PLANT_EFFECT_SD)) for p in range(1, N_PLANTS + 1)}
plant_effect = {k: v - np.mean(list(plant_effect.values())) for k, v in plant_effect.items()}

equi_rows, frailty_rows = [], []
equnr_ctr = 100000
for cls in ISO14224_CLASSES:
    p = CLASS_PARAMS[cls]
    shape = 1.0 / p["frailty_variance"]
    for _ in range(N_EQUI_PER_CLASS):
        equnr_ctr += 1
        equnr = f"EQ{equnr_ctr}"
        tplnr = iflot_df.TPLNR.iloc[int(rng.integers(0, len(iflot_df)))]
        plant = tplnr.split("-")[0]
        manufacturer = MANUFACTURERS[int(rng.integers(0, len(MANUFACTURERS)))]
        offset = int(rng.integers(0, (COMMISSION_END - OBS_START).days))
        install = as_date(offset)
        z = float(rng.gamma(shape, 1.0 / shape))          # mean 1, variance FRAILTY_VARIANCE
        equi_rows.append({
            "EQUNR": equnr, "EQTYP": cls, "TPLNR": tplnr, "IWERK": plant,
            "HERST": manufacturer, "BAUJJ": install.year, "BAUMM": install.month,
            "INSTALL_DATE": install.strftime("%Y-%m-%d"), "EQKTX": f"{cls} unit {equnr_ctr}",
        })
        # Linear predictor on the log hazard: observable effects plus unobservable frailty.
        # Equivalent to scaling eta by exp(-lp/beta). The frailty term is NOT written to EQUI -
        # putting it in the equipment master would leak the answer to the models meant to proxy it.
        lp = (mfr_effect[manufacturer]
              + plant_effect[plant]
              + CRITICALITY_COEF * (criticality_of[tplnr] - 2.5)
              + np.log(z))
        frailty_rows.append({
            "EQUNR": equnr, "EQTYP": cls, "frailty_z": round(z, 5),
            "linear_predictor": round(lp, 5),
            "mfr_effect": round(mfr_effect[manufacturer], 5),
            "plant_effect": round(plant_effect[plant], 5),
            "eta_effective_days": round(p["weibull_eta_days"] * np.exp(-lp / p["weibull_beta"]), 1),
        })

equi_df = pd.DataFrame(equi_rows)
frailty_df = pd.DataFrame(frailty_rows)
eta_effective = dict(zip(frailty_df.EQUNR, frailty_df.eta_effective_days))

mpla_rows = []
warpl_ctr = 500000
for r in equi_df.itertuples():
    warpl_ctr += 1
    mpla_rows.append({
        "WARPL": f"MP{warpl_ctr}", "EQUNR": r.EQUNR, "WARPL_ART": "CALENDAR",
        "CYCLE_DAYS": CLASS_PARAMS[r.EQTYP]["pm_cycle_days"], "PLAN_START_DATE": r.INSTALL_DATE,
    })
mpla_df = pd.DataFrame(mpla_rows)
warpl_of = dict(zip(mpla_df.EQUNR, mpla_df.WARPL))

print(f"IFLOT {len(iflot_df)} | EQUI {len(equi_df)} | MPLA {len(mpla_df)}")

# Transactional simulation

# Two processes per equipment item.

# Intrinsic renewal. Weibull residual life from the item's frailty-adjusted eta,
# competing against the calendar PM cycle. A failure is a full renewal; a PM restores only
# rho of accumulated virtual age (Kijima). This is the process the survival models in
# Section 4.2 are meant to recover.

# Defect escalation overlay. Defect notifications (M2) arrive as a Poisson process. Each
# carries a latent escalation time racing against the organisation's response delay, which
# is set by the priority code. Lose the race and the defect becomes a breakdown (M1) citing
# QMNUM_ORIG; win it and it becomes a planned repair (PM03).

# The overlay does not touch virtual age. That is the independence simplification the
# report already concedes in Section 5.4, and keeping it means escalation-driven breakdowns
# can be excluded cleanly from the survival intervals so the intrinsic Weibull process is
# recovered unpolluted. Notebook 03 does exactly that, keying on QMNUM_ORIG IS NULL.

for e in equi_df.itertuples():
    p = CLASS_PARAMS[e.EQTYP]
    beta, rho, cycle = p["weibull_beta"], p["pm_restoration_rho"], p["pm_cycle_days"]
    eta_i = eta_effective[e.EQUNR]
    base_prev = p["base_preventive_cost"]
    crit = criticality_of[e.TPLNR]
    install = float((datetime.strptime(e.INSTALL_DATE, "%Y-%m-%d") - OBS_START).days)

    # ---- intrinsic renewal ----
    virtual_age, clock, next_pm = 0.0, install, install + cycle
    while clock < HORIZON:
        failure_day = clock + residual_life(beta, eta_i, virtual_age, rng)
        if failure_day <= next_pm and failure_day < HORIZON:
            ctr["qmnum"] += 1
            qmnum = f"QM{ctr['qmnum']}"
            qmel_rows.append({
                "QMNUM": qmnum, "EQUNR": e.EQUNR, "QMART": "M1",
                "QMDAT": as_date(failure_day).strftime("%Y-%m-%d"),
                "AUSVN": as_date(failure_day).strftime("%Y-%m-%d"),
                "PRIOK": int(rng.integers(1, 3)),
                "FECOD": f"DMG-{rng.integers(1, 9)}", "URCOD": f"CAU-{rng.integers(1, 9)}",
                "QMNUM_ORIG": None,
            })
            emit_order("PM01", failure_day, e.EQUNR, e.IWERK, qmnum, None,
                       int(rng.integers(1, 3)), "BREAKDOWN", 12, 6,
                       base_prev * BREAKDOWN_COST_MULTIPLIER * float(rng.uniform(0.85, 1.25)), crit)
            virtual_age, clock = 0.0, failure_day
            next_pm = clock + cycle
        elif next_pm < HORIZON:
            emit_order("PM02", next_pm, e.EQUNR, e.IWERK, None, warpl_of[e.EQUNR],
                       3, "PREVENTIVE", 4, 1.5,
                       base_prev * float(rng.uniform(0.9, 1.1)), crit)
            virtual_age = (virtual_age + (next_pm - clock)) * (1 - rho)   # Kijima imperfect restoration
            clock = next_pm
            next_pm = clock + cycle
        else:
            break

    # ---- defect / escalation overlay ----
    t = install
    while True:
        t += float(rng.exponential(p["defect_rate_days"]))
        if t >= HORIZON:
            break
        ctr["qmnum"] += 1
        defect_id = f"QM{ctr['qmnum']}"
        prio = int(rng.choice([1, 2, 3, 4], p=[0.10, 0.25, 0.40, 0.25]))
        damage = int(rng.integers(1, 9))
        age_years = (t - install) / 365.25
        qmel_rows.append({
            "QMNUM": defect_id, "EQUNR": e.EQUNR, "QMART": "M2",
            "QMDAT": as_date(t).strftime("%Y-%m-%d"), "AUSVN": None, "PRIOK": prio,
            "FECOD": f"DMG-{damage}", "URCOD": f"CAU-{rng.integers(1, 9)}", "QMNUM_ORIG": None,
        })
        # Escalation hazard: severe damage codes, critical locations and older assets
        # deteriorate faster. These are exactly the features the prioritizer sees.
        lam = (p["escalation_base_hazard"]
               * (1 + 0.35 * (damage >= 6))
               * (1 + 0.15 * crit)
               * (1 + 0.10 * age_years))
        t_escalate = float(rng.exponential(1.0 / lam))
        mean_resp = PRIORITY_RESPONSE_DAYS[prio]
        t_respond = max(1.0, float(rng.normal(mean_resp, mean_resp * 0.4)))

        if t_escalate < t_respond and t + t_escalate < HORIZON:
            ctr["qmnum"] += 1
            breakdown_id = f"QM{ctr['qmnum']}"
            day = t + t_escalate
            qmel_rows.append({
                "QMNUM": breakdown_id, "EQUNR": e.EQUNR, "QMART": "M1",
                "QMDAT": as_date(day).strftime("%Y-%m-%d"),
                "AUSVN": as_date(day).strftime("%Y-%m-%d"), "PRIOK": 1,
                "FECOD": f"DMG-{damage}", "URCOD": f"CAU-{rng.integers(1, 9)}",
                "QMNUM_ORIG": defect_id,
            })
            emit_order("PM01", day, e.EQUNR, e.IWERK, breakdown_id, None, 1, "BREAKDOWN", 12, 6,
                       base_prev * BREAKDOWN_COST_MULTIPLIER * float(rng.uniform(0.85, 1.25)), crit)
        elif t + t_respond < HORIZON:
            emit_order("PM03", t + t_respond, e.EQUNR, e.IWERK, defect_id, None, prio,
                       "PLANNED_REPAIR", 6, 2.5,
                       base_prev * PLANNED_REPAIR_MULTIPLIER * float(rng.uniform(0.85, 1.15)), crit)

qmel_df = pd.DataFrame(qmel_rows)
aufk_df = pd.DataFrame(aufk_rows)
afru_df = pd.DataFrame(afru_rows)
coss_df = pd.DataFrame(coss_rows)
params_df = pd.DataFrame([{"EQTYP": c, **v} for c, v in CLASS_PARAMS.items()])

print("simulation complete")

# Sanity summary

intrinsic = qmel_df[(qmel_df.QMART == "M1") & (qmel_df.QMNUM_ORIG.isna())]
defects = qmel_df[qmel_df.QMART == "M2"]
escalated = qmel_df[qmel_df.QMNUM_ORIG.notna()]
med = coss_df.groupby("COST_TYPE").TOTAL_ACTUAL_COST.median()

print(f"IFLOT {len(iflot_df):>6,}   EQUI {len(equi_df):>6,}   MPLA {len(mpla_df):>6,}")
print(f"QMEL  {len(qmel_df):>6,}   {dict(qmel_df.QMART.value_counts())}")
print(f"AUFK  {len(aufk_df):>6,}   {dict(aufk_df.AUART.value_counts())}")
print(f"AFRU  {len(afru_df):>6,}   COSS {len(coss_df):>6,}")
print(f"total {sum(map(len, [iflot_df, equi_df, mpla_df, qmel_df, aufk_df, afru_df, coss_df])):,}\n")
print(f"intrinsic failures      {len(intrinsic):,}  (feed the survival models)")
print(f"escalation-driven M1    {len(escalated):,}  ({len(escalated)/len(defects):.1%} of defects)")
print(f"breakdown:preventive    {med.get('BREAKDOWN', 0) / med.get('PREVENTIVE', 1):.2f}x  (target {BREAKDOWN_COST_MULTIPLIER})")

thin = intrinsic.merge(equi_df[["EQUNR", "EQTYP"]], on="EQUNR").EQTYP.value_counts()
print(f"\nintrinsic failures per class (min {thin.min()}):")
print(thin.sort_values().to_string())

# Write to the volume

# Entities to landing/, ground truth to ground_truth/. Two folders, deliberately —
# nothing downstream can pick up the generating parameters through a wildcard read.

for path in [LANDING, GROUND_TRUTH]:
    dbutils.fs.mkdirs(path)

for name, df in [("IFLOT", iflot_df), ("EQUI", equi_df), ("MPLA", mpla_df),
                 ("QMEL", qmel_df), ("AUFK", aufk_df), ("AFRU", afru_df), ("COSS", coss_df)]:
    df.to_csv(f"{LANDING}/{name}.csv", index=False)
    print(f"wrote {LANDING}/{name}.csv  ({len(df):,} rows)")

params_df.to_csv(f"{GROUND_TRUTH}/simulation_parameters.csv", index=False)
frailty_df.to_csv(f"{GROUND_TRUTH}/equipment_frailty.csv", index=False)
pd.DataFrame(
    [{"term": f"manufacturer:{k}", "log_hazard_effect": round(v, 5)} for k, v in mfr_effect.items()]
    + [{"term": f"plant:{k}", "log_hazard_effect": round(v, 5)} for k, v in plant_effect.items()]
    + [{"term": "criticality (per step)", "log_hazard_effect": CRITICALITY_COEF}]
).to_csv(f"{GROUND_TRUTH}/covariate_effects.csv", index=False)
print(f"\nwrote {GROUND_TRUTH}/simulation_parameters.csv  ({len(params_df)} classes)")
print(f"wrote {GROUND_TRUTH}/equipment_frailty.csv  ({len(frailty_df):,} items)")

display(dbutils.fs.ls(LANDING))

# Now run 04_bronze_ingest → 05_silver_conform → 06_gold_survival →
# 07_quality_and_exports.

# Then update Section 4.1.3 of the report from this run's actual counts. The table there
# still carries v2 figures and a [TO VERIFY] flag. Take the numbers from the output above
# rather than tuning the generator to match numbers written before it ran.